In [1]:
# Librerias
import warnings

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import pandas as pd
from pathlib import Path
import datetime as dt
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json

from google.cloud import bigquery
from google.oauth2 import service_account

delta=dt.timedelta(days=1)

In [2]:
def Precios_Tebsa(s_FechaIni):
    # Información del proyecto y autenticación a BQ
    project_id = "enersinc-tbsg-bq"
    key_path = "C:\BigQuery\cohesive-point-249003-a049682d1f98.json"

    # Cargar las credenciales del archivo JSON
    credentials = service_account.Credentials.from_service_account_file(key_path)

    # Crear el cliente de BigQuery
    client = bigquery.Client(project=project_id, credentials=credentials)

    d_FechaIni=dt.datetime.strptime(s_FechaIni, '%d/%m/%Y').strftime('%Y-%m-%d')
    d_FechaIni = str(d_FechaIni)

    # Consulta a la maestra de recursos
    query = """
    select t1.planta, t1.concepto, t1.precio
    from 
    (select planta, concepto, hora1/1000 as precio, IF(LENGTH(SUBSTR(concepto, 2)) > 0, SAFE_CAST(SUBSTR(concepto, 2) AS INT64), 1) AS concepto_num
    from tbsg.private_oferta_ori
    where fechaoperacion = 'd_FechaIni' and concepto like 'P%') t1
    order by t1.planta, t1.concepto_num 
    """

    query = query.replace('d_FechaIni', d_FechaIni)

    # Ejecutar la consulta
    df_Precio = client.query(query).to_dataframe()

    return df_Precio

In [3]:

# Ruta del archivo
# script_dir = Path(__file__).resolve()
# script_dir=script_dir.parent.parent.parent
# sPathfile=os.path.join(script_dir,r"Modules\Utils\ArchivosAux",sFile)
sPathFolder=r"C:\Alejo\cops\Modules\Utils\ArchivosAux"

sFile=r"Parametros.json"
sPathfile=os.path.join(sPathFolder,sFile)

# Open and load the JSON file
with open(sPathfile,'r') as f:
    data = json.load(f)

# Almancenar los parámetros en variables python
year=data['Parametros']['Ano']
mes=data['Parametros']['Mes']
Carpeta=data['Parametros']['Carpeta']
FechaInicial=data['Parametros']['Fecha_Inicial']
FechaFinal=data['Parametros']['Fecha_Final']
FileName=data['Parametros']['Nombre_Archivo']

df_Precio =Precios_Tebsa(FechaInicial)

In [4]:
df_Precio.to_csv(os.path.join(r"C:\Alejo\Eje de Planeación\Análisis Energético\Ejecuciones 30 días\\" + f"{int(year):00d}" + "-" + f"{int(mes):02d}" ,Carpeta , f"Precios_Tebsa_{year}_{mes}.csv"), index=False)
df_Precio

,planta,concepto,precio
0,BARRANQUILLA3,P,1203.212
1,BARRANQUILLA4,P,1196.390
2,TEBSABCC,P1,1644.364
3,TEBSABCC,P2,794.777
4,TEBSABCC,P3,1832.656
5,TEBSABCC,P4,812.004
6,TEBSABCC,P5,1693.545
7,TEBSABCC,P6,855.331
8,TEBSABCC,P7,717.881
9,TEBSABCC,P8,1577.712
